# 🏗️ Notebook 1: Collaborative Whiteboard — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/collaborative-whiteboard
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A real-time whiteboard. Multiple users draw shapes on one canvas; edits appear on everyone's screen within ~100ms. Eventual consistency: temporary disagreements are fine, but everyone must converge to the same final state.

## Requirements

### Functional
- Draw/move/delete shapes in real time.
- See other users' cursors & selections.
- Late joiners load current state.
- Offline edits merge back when online.

### Non-functional
- Convergence: all clients end up with the same canvas.
- Fan-out latency < 100ms.
- Tolerate brief network partitions.

## Back-of-envelope

- 100k concurrent boards × 5 users avg = 500k WS connections.
- Per edit: 300 B. 10 edits/s/board × 100k = 1M msg/s → needs sharded WS gateway.

## High-level architecture

```
  [Browser] ──WebSocket──► WS Gateway ──► Room Service
                                            │
                 ┌──────────────────────────┼────────┐
                 ▼                          ▼        ▼
              Redis pub/sub         CRDT store   Snapshot store
                 │                   (per-board)   (S3)
                 ▼
              other WS Gateways (cross-node fan-out)
```

- **Rooms** are sharded by board_id.
- **Snapshots** taken periodically so new joiners don't replay full op log.

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.